# Setup

In [ ]:
!pip install requests beautifulsoup4 lxml

In [ ]:
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

# Translation Agent

## Commentary Parser

In [ ]:
import re
import requests
from bs4 import BeautifulSoup

def parse_enduring_word(book: str, chapter: int, start_verse: int, end_verse: int):
    """Scrapes Enduring Word commentary, isolates target boundaries,

    and returns a clean list of structured blocks (No console prints).
    """
    book_url = book.lower().replace(" ", "-")
    url = f"https://enduringword.com/bible-commentary/{book_url}-{chapter}/"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"❌ HTTP Error {response.status_code}")
        return []

    soup = BeautifulSoup(response.content, "lxml")
    content_area = soup.find("div", class_="entry-content") or soup.find(
        "article"
    )
    if not content_area:
        print("❌ Could not find the core article content area.")
        return []

    current_verse_context = None
    structured_blocks = []
    pending_h3 = None

    for element in content_area.find_all(["h3", "h4", "p"]):
        # 1. Skip the main Bible text blocks entirely
        if (
            element.name == "p"
            and element.has_attr("class")
            and "ew-bible-text" in element["class"]
        ):
            continue

        text = element.get_text().strip()
        if not text:
            continue

        # 2. --- MAIN SECTION HEADERS (H3) ---
        if element.name == "h3" and not ("(" in text and ")" in text):
            pending_h3 = text
            continue

        # 3. --- SCRIPTURE ANCHORS (H4 / H3 with parentheses) ---
        if element.name == "h4" or (
            element.name == "h3" and "(" in text and ")" in text
        ):
            verse_match = re.search(
                r"\((\d+)[a-zA-Z]?(?:\s*-\s*(\d+)[a-zA-Z]?)?\)", text
            )
            if verse_match:
                v_start = int(verse_match.group(1))
                v_end = (
                    int(verse_match.group(2)) if verse_match.group(2) else v_start
                )

                current_verse_context = (v_start, v_end)

                # Lower limit boundary break
                if v_start > end_verse:
                    break

                # Append matching section layout headings
                if v_end >= start_verse and v_start <= end_verse:
                    if pending_h3:
                        structured_blocks.append(
                            {
                                "verse_range": current_verse_context,
                                "type": "main_section_header",
                                "bold_quote": "",
                                "full_text": pending_h3,
                            }
                        )
                        pending_h3 = None

                    structured_blocks.append(
                        {
                            "verse_range": current_verse_context,
                            "type": "scripture_anchor",
                            "bold_quote": "",
                            "full_text": text,
                        }
                    )
                continue

        # 4. --- PARSE BODY PARAGRAPHS & OUTLINE HOOKS (p) ---
        if element.name == "p":
            if re.search(r"©1996.*Enduring\s+Word\s+Bible\s+Commentary", text):
                continue

            bold_tag = element.find("strong")
            bold_quote = bold_tag.get_text().strip() if bold_tag else ""

            if bold_quote and bold_quote == text:
                continue

            if (
                current_verse_context is None
                or current_verse_context[1] < start_verse
            ):
                continue

            v_start, v_end = current_verse_context
            if v_start > end_verse:
                break

            structured_blocks.append(
                {
                    "verse_range": current_verse_context,
                    "type": "body_paragraph",
                    "bold_quote": bold_quote,
                    "full_text": text,
                }
            )

    return structured_blocks


def print_commentary_cache(structured_blocks: list):
    """Takes a local cache array of structured blocks and prints them

    with beautiful hierarchical tree indentation.
    """
    if not structured_blocks:
        print("⚠️ Local cache block array is empty.")
        return

    print(
        f"🖥️ Rendering {len(structured_blocks)} structured elements from local memory cache:\n"
    )

    for block in structured_blocks:
        text = block["full_text"]
        b_type = block["type"]

        if b_type == "main_section_header":
            print(f"\n🟢 [H3] {text}")

        elif b_type == "scripture_anchor":
            print(f"  🔵 [H4 Anchor] {text}")

        elif b_type == "body_paragraph":
            # Identify line-nesting structures for formatting lookups
            is_roman_numeral = re.match(
                r"^(v|i|x)+\.", text, re.IGNORECASE
            )  # i., ii., etc.
            is_alpha_letter = re.match(r"^[a-z]\.", text)  # a., b., c., etc.

            if is_roman_numeral:
                print(f"      🔹 [Sub-point] {text}")
            elif is_alpha_letter:
                print(f"    🔸 [Point] {text}")
            else:
                print(f"    ... [Standard Body] {text}")

## Bible Parser

In [ ]:
BOOK_CODE_MAP = {
    "genesis": "sa",
    "exodus": "xu",
    "leviticus": "le",
    "numbers": "dan",
    "deuteronomy": "phu",
    "joshua": "gios",
    "judges": "cac",
    "ruth": "ru",
    "1 samuel": "1sa",
    "2 samuel": "2sa",
    "1 kings": "1vua",
    "2 kings": "2vua",
    "1 chronicles": "1su",
    "2 chronicles": "2su",
    "ezra": "exo",
    "nehemiah": "ne",
    "esther": "et",
    "job": "giop",
    "psalms": "thi",
    "proverbs": "ch",
    "ecclesiastes": "tr",
    "song of solomon": "nha",
    "isaiah": "es",
    "jeremiah": "gie",
    "lamentations": "ca",
    "ezekiel": "exe",
    "daniel": "da",
    "hosea": "os",
    "joel": "gio",
    "amos": "am",
    "obadiah": "ap",
    "jonah": "gion",
    "micah": "mi",
    "nahum": "na",
    "habakkuk": "ha",
    "zephaniah": "so",
    "haggai": "ag",
    "zechariah": "xa",
    "malachi": "ma",

    "matthew": "mat",
    "mark": "mac",
    "luke": "lu",
    "john": "gi",
    "acts": "cong",
    "romans": "ro",
    "1 corinthians": "1co",
    "2 corinthians": "2co",
    "galatians": "ga",
    "ephesians": "eph",
    "philippians": "phi",
    "colossians": "co",
    "1 thessalonians": "1te",
    "2 thessalonians": "2te",
    "1 timothy": "1ti",
    "2 timothy": "2ti",
    "titus": "tit",
    "philemon": "phil",
    "hebrews": "he",
    "james": "gia",
    "1 peter": "1phi",
    "2 peter": "2phi",
    "1 john": "1gi",
    "2 john": "2gi",
    "3 john": "3gi",
    "jude": "giu",
    "revelation": "kh"
}

VI_TO_EN_BOOKS = {
    "Ê-SAI": "Isaiah",

    "MA-THI-Ơ": "Matthew", "MÁC": "Mark", "LU-CA": "Luke", "GIĂNG": "John",
    "CÔNG-VỤ": "Acts", "RÔ-MA": "Romans", "I CÔ-RINH-TÔ": "1 Corinthians",
    "II CÔ-RINH-TÔ": "2 Corinthians", "GA-LA-TI": "Galatians", "Ê-PHÊ-SÔ": "Ephesians",
    "PHI-LÍP": "Philippians", "CÔ-LÔ-SE": "Colossians", "I TÊ-SA-LÔ-NI-CA": "1 Thessalonians",
    "II TÊ-SA-LÔ-NI-CA": "2 Thessalonians", "I TI-MÔ-THÊ": "1 Timothy",
    "II TI-MÔ-THÊ": "2 Timothy", "TÍT": "Titus", "PHI-LÊ-MÔN": "Philemon",
    "HÊ-BƠ-RƠ": "Hebrews", "GIA-CƠ": "James", "I PHI-E-RƠ": "1 Peter",
    "II PHI-E-RƠ": "2 Peter", "I GIĂNG": "1 John", "II GIĂNG": "2 John",
    "III GIĂNG": "3 John", "GIU-ĐE": "Jude", "KHẢI-HUYỀN": "Revelation"
}

In [ ]:
import re
import requests
from bs4 import BeautifulSoup


def parse_httlvn_chapter_optimized(book_name: str, chapter: int, start_verse: int, end_verse: int, version: str):
    """
    Scrapes a target range of verses for a given version from kinhthanh.httlvn.org.
    Cleans out anchor symbols (⚓) and bypasses structural section titles.
    """
    # 1. Map English book string to platform code automatically
    book_clean = book_name.strip().lower()
    book_code = BOOK_CODE_MAP.get(book_clean)

    if not book_code:
        print(f"❌ Map Error: '{book_name}' not found in BOOK_CODE_MAP database.")
        return {}

    url = f"https://kinhthanh.httlvn.org/doc-kinh-thanh/{book_code}/{chapter}?v={version}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"❌ Error loading {version} from platform. Code: {response.status_code}")
        return {}

    soup = BeautifulSoup(response.content, "lxml")
    content_area = soup.find("div", class_="bible-read")
    if not content_area:
        print(f"❌ Could not find 'bible-read' container for version {version}")
        return {}

    # Strip out all internal title/heading blocks immediately
    for title_div in content_area.find_all(["div", "span"], class_="title"):
        title_div.decompose()

    verse_data = {}
    verse_class_pattern = re.compile(rf"verse\s+{book_code}_{chapter}_\d+")

    for span in content_area.find_all("span", class_=verse_class_pattern):
        sup_tag = span.find("sup")
        if not sup_tag:
            continue

        v_num_text = sup_tag.get_text().strip()
        if not v_num_text.isdigit():
            continue
        v_num = int(v_num_text)

        # BOUNDARY OPTIMIZATION: Ignore this verse completely if it falls outside our targeted range
        if v_num < start_verse or v_num > end_verse:
            continue

        # Remove tooltips and internal anchor text blocks completely
        for anchor in span.find_all("a", class_="data-toggle"):
            anchor.decompose()

        # Extract remaining textual nodes
        raw_text = span.get_text().strip()

        # Remove the leading verse number string prefix
        clean_text = raw_text[len(v_num_text):].strip()

        # CLEANUP UPGRADE: Remove the anchor symbol string explicitly if it persists in text fields
        clean_text = clean_text.replace("⚓", "").strip()

        # Normalize trailing/consecutive whitespace characters
        clean_text = re.sub(r"\s+", " ", clean_text).replace("\xa0", " ")

        if clean_text:
            verse_data[v_num] = clean_text

    return verse_data


def build_bounded_language_cache(book_name: str, chapter: int, start_verse: int, end_verse: int):
    """
    Combines both parsed structures into a unified, range-restricted local memory lookup cache.
    """
    print(f"📡 Bounded Fetch -> Vietnamese (VI1934) for Verses {start_verse}-{end_verse}...")
    vi_verses = parse_httlvn_chapter_optimized(book_name, chapter, start_verse, end_verse, "VI1934")

    print(f"📡 Bounded Fetch -> English (NKJV) for Verses {start_verse}-{end_verse}...")
    en_verses = parse_httlvn_chapter_optimized(book_name, chapter, start_verse, end_verse, "NKJV")

    unified_cache = {}
    all_verse_numbers = set(list(vi_verses.keys()) + list(en_verses.keys()))

    for num in sorted(all_verse_numbers):
        unified_cache[num] = {
            "eng": en_verses.get(num, ""),
            "vie": vi_verses.get(num, "")
        }

    print(f"✅ Bounded alignment complete! Stored {len(unified_cache)} target verses in memory.")
    return unified_cache

## Parsing Pipeline

In [ ]:
def dynamic_bible_pipeline(book_name: str, chapter: int, user_start: int, user_end: int):
    """
    Orchestrates the entire data ingestion. It runs the commentary parser first,
    discovers the true structural boundaries, and dynamically expands the Bible
    cache fetch to match.
    """
    print(f"🎬 Starting Pipeline for {book_name} {chapter}:{user_start}-{user_end}")
    print("------------------------------------------------------------------------")

    # 1. Run the commentary parser with the user's requested scope
    commentary_blocks = parse_enduring_word(book_name, chapter, user_start, user_end)

    if not commentary_blocks:
        print("❌ No commentary blocks extracted. Aborting pipeline.")
        return None, None

    # 2. Find the minimum and maximum verses actually touched by the commentary
    all_touched_verses = []
    for block in commentary_blocks:
        v_start, v_end = block["verse_range"]
        all_touched_verses.extend([v_start, v_end])

    # Determine the true structural bounds computed by the commentary layout
    true_start = min(all_touched_verses)
    true_end = max(all_touched_verses)

    print(f"\n🔄 Boundary Adjustment Verified:")
    print(f"   ↳ User requested: Verses {user_start} - {user_end}")
    print(f"   ↳ Commentary requires: Verses {true_start} - {true_end} (to keep sections complete)")

    # 3. Fetch the Bible reference text using the newly adjusted true boundaries
    optimized_bible_cache = build_bounded_language_cache(book_name, chapter, true_start, true_end)

    print("\n🚀 Pipeline Successfully Synced and Enriched!")
    print("------------------------------------------------------------------------")
    return commentary_blocks, optimized_bible_cache

## Google Docs API Integration

In [ ]:
import os
from google.colab import auth
import google.auth
from googleapiclient.discovery import build

# 2. Trigger the Google pop-up verification screen
auth.authenticate_user()

# 3. Grab your session credentials
credentials, project_id = google.auth.default()

# 4. Initialize the official Google Docs API engine
docs_service = build('docs', 'v1', credentials=credentials)
print("✨ Google Docs API successfully authorized!")

# Data Curation

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

# 1. Authenticate your user account
auth.authenticate_user()

# 2. Build the Docs service (which you already have)
docs_service = build('docs', 'v1')

# 3. Build the missing Drive service to scan folders
drive_service = build('drive', 'v3')
print("✅ Both docs_service and drive_service are initialized and ready!")

In [ ]:
# Insert the folder ID containing your past completed study guides
PAST_TRANSLATIONS_FOLDER_ID = "YOUR_DRIVE_FOLDER_ID"

def scan_drive_for_samples(folder_id):
    query = f"'{folder_id}' in parents and mimeType = 'application/vnd.google-apps.document' and trashed = false"

    results = drive_service.files().list(
        q=query,
        pageSize=100,
        fields="nextPageToken, files(id, name)"
    ).execute()

    files = results.get('files', [])

    print(f"📁 Analysis Complete: Found {len(files)} completed documents in your archive.")
    print("=" * 75)
    for idx, file in enumerate(files, start=1):
        print(f"  [{idx}] 📄 Name: {file['name']:<30} | ID: {file['id']}")

    return files

# Run the scanner
archived_files = scan_drive_for_samples(PAST_TRANSLATIONS_FOLDER_ID)

In [ ]:
def build_raw_training_dataset(files_list):
    dataset_samples = []

    for file in files_list:
        print(f"📥 Pulling text data from: {file['name']}...")

        # Reusing your working Google Docs GET setup
        doc = docs_service.documents().get(documentId=file['id']).execute()
        doc_content = doc.get('body').get('content')

        # Rebuilding your paragraph parser to collect sentences sequentially
        paragraphs = []
        for element in doc_content:
            if 'paragraph' in element:
                p_text = "".join([
                    run.get('textRun', {}).get('content', '')
                    for run in element['paragraph']['elements']
                ])
                if p_text.strip():
                    paragraphs.append(p_text.strip())

        # Store the structured file contents
        dataset_samples.append({
            "source_doc": file['name'],
            "paragraphs": paragraphs
        })

    return dataset_samples

# Extract texts into memory arrays
extracted_raw_data = build_raw_training_dataset(archived_files)
print(f"\n✅ Ready! Captured matching text arrays from all available files.")

In [ ]:
import re

def parse_reference_from_header(paragraphs):
    """
    Scans the top elements of your text array to extract the English lookup parameters.
    Example input: 'GIẢI KINH MA-THI-Ơ 19:3-12' -> Output: ('Matthew', 19, 3, 12)
    """
    # Scan the first few rows for the layout keyword
    for text in paragraphs[:4]:
        if "GIẢI KINH" in text:
            # Matches: BOOK_NAME Chapter:Verse_Start-Verse_End
            pattern = r"GIẢI KINH\s+((?:\d\s+)?[A-ZĂÁÂĐÊÔƠƯ\- ]+)\s+(\d+):(\d+)\-(\d+)"
            match = re.search(pattern, text)
            if match:
                vi_book, chapter, v_start, v_end = match.groups()
                en_book = VI_TO_EN_BOOKS.get(vi_book.strip())
                if en_book:
                    return {
                        "book": en_book,
                        "chapter": int(chapter),
                        "verse_start": int(v_start),
                        "verse_end": int(v_end)
                    }
    return None

In [ ]:
import json
import re

def process_and_export_all_datasets(extracted_raw_data):
    print(f"🎬 Compiling final fine-tuning dataset layers...")

    t1_count = 0
    t2_count = 0

    with open("task1_extraction.jsonl", "w", encoding="utf-8") as t1_f, \
         open("task2_translation.jsonl", "w", encoding="utf-8") as t2_f:

        for doc_item in extracted_raw_data:
            doc_name = doc_item["source_doc"]
            paragraphs = doc_item["paragraphs"]

            # 1. Pull the English structural reference keys from the header line
            ref = parse_reference_from_header(paragraphs)
            if not ref:
                print(f"⚠️ Skipping {doc_name}: No structural reference header found.")
                continue

            print(f"📖 Aligned {doc_name} -> {ref['book']} {ref['chapter']}:{ref['verse_start']}-{ref['verse_end']}")

            # 2. RUN YOUR UNIFIED PIPELINE
            try:
                parsed_english_blocks, bible_cache = dynamic_bible_pipeline(
                    book_name=ref["book"],
                    chapter=ref["chapter"],
                    user_start=ref["verse_start"],
                    user_end=ref["verse_end"]
                )

                if not parsed_english_blocks or not bible_cache:
                    print(f"⚠️ Skipping {doc_name}: Pipeline returned empty data arrays.")
                    continue

            except Exception as e:
                print(f"❌ Execution failed inside dynamic_bible_pipeline for {doc_name}: {str(e)}")
                print()
                continue

            # 3. COMBINE PARSED DATA WITH YOUR ARCHIVED VIETNAMESE ENTRIES
            meaningful_vi_paragraphs = [p for p in paragraphs if not ("NGHIÊN CỨU" in p or "GIẢI KINH" in p or "CỦA ENDURING" in p)]

            # Zip aligned English blocks 1:1 right alongside your Vietnamese paragraphs
            for eng_block, vi_text in zip(parsed_english_blocks, meaningful_vi_paragraphs):
                v_start, v_end = eng_block["verse_range"]

                en_full_text = eng_block["full_text"].strip()
                vi_text_clean = vi_text.strip()

                # ─── AUTOMATICALLY EXTRACT PREFIX FROM ENGLISH TEXT ───
                # Matches patterns like 'a.', 'b.', 'i.', 'ii.' at the absolute beginning of the English text
                prefix_match = re.match(r"^([a-z]\.|i{1,3}\.|iv\.|v\.)\s*", en_full_text, re.IGNORECASE)

                if prefix_match:
                    prefix_str = prefix_match.group(1) # e.g., "a." or "i."

                    # Check if the Vietnamese text already starts with this prefix
                    escaped_prefix = re.escape(prefix_str)
                    if not re.match(rf"^{escaped_prefix}", vi_text_clean, re.IGNORECASE):
                        ground_truth_translation = f"{prefix_str} {vi_text_clean}".strip()
                    else:
                        ground_truth_translation = vi_text_clean
                else:
                    # No list prefix detected in English block
                    ground_truth_translation = vi_text_clean

                # Core Verse Matching Line from your newly returned optimized cache
                english_verse_line = bible_cache.get(v_start, {}).get("eng", "")
                vietnamese_verse_line = bible_cache.get(v_start, {}).get("vie", "")

                # Extract matching quote anchors to train Task 1
                quotes_found = re.findall(r'“([^”]+)”', vi_text_clean)
                extracted_vi_quote = quotes_found[0].strip().rstrip(":") if quotes_found else ""

                # Sanitize the companion English bold anchor quote for explicit targeting mapping
                raw_eng_quote = eng_block.get("bold_quote", "").strip().rstrip(":")

                # WRITE TASK 1
                if raw_eng_quote and english_verse_line:
                    t1_record = {
                        "messages": [
                            {"role": "system", "content": "You are a bilingual scripture matching assistant. Your job is to extract the exact phrase from the Vietnamese translation that matches the English quote."},
                            {"role": "user", "content": f"English Verse: {english_verse_line}\nVietnamese Verse: {vietnamese_verse_line}\nEnglish Quote to Match: {raw_eng_quote}\n\n/no_think Extract and output ONLY the corresponding Vietnamese phrase from the Vietnamese Verse."},
                            {"role": "assistant", "content": extracted_vi_quote}
                        ]
                    }
                    t1_f.write(json.dumps(t1_record, ensure_ascii=False) + "\n")
                    t1_count += 1

                # WRITE TASK 2
                if extracted_vi_quote and raw_eng_quote:
                    quote_instruction = f"CRITICAL: You must use the exact phrase '{extracted_vi_quote}' for '{raw_eng_quote}' in this text.\n"
                else:
                    quote_instruction = ""

                t2_record = {
                    "messages": [
                        {"role": "system", "content": "You are a professional theological translator. Your job is to translate English commentary into smooth, traditional Vietnamese biblical study text."},
                        {"role": "user", "content": f"/no_think\n{quote_instruction}Translate this commentary text to Vietnamese (keep list markers like 'a.', 'b.', 'i.' intact):\n{en_full_text}"},
                        {"role": "assistant", "content": ground_truth_translation}
                    ]
                }
                t2_f.write(json.dumps(t2_record, ensure_ascii=False) + "\n")
                t2_count += 1

    print("=" * 75)
    print(f"✨ Compilation Complete! Files compiled and ready for training scripts:")
    print(f"   • {t1_count} entries -> task1_extraction.jsonl")
    print(f"   • {t2_count} entries -> task2_translation.jsonl")

In [ ]:
process_and_export_all_datasets(extracted_raw_data)

In [ ]:
def check_dataset_alignment_health(extracted_raw_data):
    print("🔍 RUNNING DATASET ALIGNMENT HEALTH CHECK...\n")
    print(f"{'Document Name':<30} | {'ENG Blocks':<10} | {'VIE Blocks':<10} | {'Status':<10}")
    print("-" * 70)

    broken_files = 0
    perfect_files = 0

    for doc_item in extracted_raw_data:
        doc_name = doc_item["source_doc"]
        paragraphs = doc_item["paragraphs"]

        ref = parse_reference_from_header(paragraphs)
        if not ref:
            print(f"⚠️ {doc_name:<28} | --         | --         | MISSING REF HEADER")
            continue

        try:
            parsed_english_blocks, _ = dynamic_bible_pipeline(
                book_name=ref["book"],
                chapter=ref["chapter"],
                user_start=ref["verse_start"],
                user_end=ref["verse_end"]
            )

            meaningful_vi_paragraphs = [p for p in paragraphs if not ("NGHIÊN CỨU" in p or "GIẢI KINH" in p or "CỦA ENDURING" in p)]

            eng_len = len(parsed_english_blocks) if parsed_english_blocks else 0
            vi_len = len(meaningful_vi_paragraphs)

            if eng_len == vi_len:
                status = "✅ PERFECT"
                perfect_files += 1
            else:
                status = f"❌ MISMATCH ({eng_len - vi_len:+})"
                broken_files += 1

            print(f"{doc_name:<30} | {eng_len:<10} | {vi_len:<10} | {status}")

        except Exception as e:
            print(f"💥 {doc_name:<28} | ERROR      | --         | {str(e)[:20]}")

    print("=" * 70)
    print(f"📊 REPORT: {perfect_files} files are perfectly aligned. {broken_files} files need fixing.")

# Run the health check
check_dataset_alignment_health(extracted_raw_data)

In [ ]:
def debug_single_file_alignment(target_doc_name, extracted_raw_data):
    """
    Prints a side-by-side comparison of English text vs Vietnamese paragraphs
    to visually spot where the index alignment drifted.
    """
    for doc_item in extracted_raw_data:
        if doc_item["source_doc"] != target_doc_name:
            continue

        paragraphs = doc_item["paragraphs"]
        ref = parse_reference_from_header(paragraphs)

        if not ref:
            print("❌ Could not parse Bible reference header for this file.")
            return

        # Run your pipeline to get the true english blocks
        parsed_english_blocks, _ = dynamic_bible_pipeline(
            book_name=ref["book"], chapter=ref["chapter"],
            user_start=ref["verse_start"], user_end=ref["verse_end"]
        )

        meaningful_vi_paragraphs = [p for p in paragraphs if not ("NGHIÊN CỨU" in p or "GIẢI KINH" in p or "CỦA ENDURING" in p)]

        print(f"🛠️ DEBUGGING ALIGNMENT FOR: {target_doc_name}")
        print(f"🇬🇧 English Blocks: {len(parsed_english_blocks)} | 🇻🇳 Vietnamese Paragraphs: {len(meaningful_vi_paragraphs)}")
        print("=" * 100)
        print(f"{'Index':<5} | {'[ENG] Original English Source snippet':<40} | {'[VIE] Your Translated Document snippet':<40}")
        print("-" * 100)

        max_len = max(len(parsed_english_blocks), len(meaningful_vi_paragraphs))
        for i in range(max_len):
            eng_snap = ""
            vi_snap = ""

            if i < len(parsed_english_blocks):
                # Pull snippet text clean from the object dict entry
                eng_snap = parsed_english_blocks[i]["full_text"].replace('\n', ' ').strip()[:38]
            if i < len(meaningful_vi_paragraphs):
                vi_snap = meaningful_vi_paragraphs[i].replace('\n', ' ').strip()[:38]

            # If they don't look like they match conceptually, add a warning flag indicator
            flag = "  " if (eng_snap and vi_snap) else "⚠️"

            print(f"{i:<5} {flag} | {eng_snap:<60} | {vi_snap:<60}")
        return

In [ ]:
debug_single_file_alignment("LESSON 32 + 33 + 34 + 35 + 36", extracted_raw_data)

In [ ]:
debug_single_file_alignment("LESSON 72", extracted_raw_data)